In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Compute relative intensity
def normalize_intensities(intensities):
  """
  Argument:
    intensities: List of intensities.
  """
  max_intensity = max(intensities)
  if max_intensity > 0:
    return [val / max_intensity for val in intensities]
  else:
    return [0.0] * len(intensities)

In [3]:
def process_fragment_and_intensities_strings(fragments_string, intensities_string):
  """
  Process fragment types and intensities strings.
  Argument:
    fragments_string: String containing fragment types separated by ';'.
    intensities_string: String containing intensities separated by ';'.
  """
  fragment_types = fragments_string.split(";")
  intensities = intensities_string.split(";")

  cleaned_fragment_types = []
  cleaned_intensities = []

  for i, fragment_type in enumerate(fragment_types):
    # Filter out fragment types containing '-', 'a', 'c', 'x', or 'z'
    # "-" is contained in neutral loss fragment (like -H2O -NH3)
      if not any(char in fragment_type for char in ['-', 'a', 'c', 'x', 'z']):
        if fragment_type.count("+") == 0: # If don't have charge at the end of str
          cleaned_fragment_types.append(fragment_type + ("(1+)"))
        else:
          cleaned_fragment_types.append(fragment_type)
        # Convert intensity to numeric type as elements of [intensities] are still strings
        cleaned_intensities.append(float(intensities[i]))
  return cleaned_fragment_types, cleaned_intensities


In [4]:
def name_fragment_columns():  
  columns = []
  for fragment_charge in ["1", "2"]:
    for fragment_type in ["b", "y"]:
      for i in range(1, 40):
        columns.append(fragment_type + str(i) + "(" + fragment_charge + "+)")
  return columns

In [24]:
# Pack data rows into a list before creating a dataframe
def create_data_entries(dataframe, columns):
  rows_data = []
  for _, row in dataframe.iterrows():
    row_dict = {}

    # Initialize all fragment ion columns to 0 for the row
    for col in columns:
        row_dict[col] = 0.0

    fragment_types, intensities = process_fragment_and_intensities_strings(
        row["Matches"],
        row["Intensities"]
        )

    # Normalize intensities if not empty
    if intensities:
        normalized_intensities = normalize_intensities(intensities)
    else:
        normalized_intensities = []

    # Populate fragment ion columns in the row_dict
    for i, fragment_type in enumerate(fragment_types):
          row_dict[fragment_type] = normalized_intensities[i]

    rows_data.append(row_dict)
  return rows_data


In [28]:
DATA_DIR = 'final_data'
SAVE_DIR = 'final_data'
fragment_columns = name_fragment_columns()

lookup_df = pd.read_csv(os.path.join(DATA_DIR, "lookup.tsv"), sep='\t')
files_to_process = lookup_df['Reference table'].drop_duplicates().tolist()
for file in files_to_process:
    precursors_df = pd.read_csv(os.path.join(DATA_DIR, file), sep='\t')
    data_entries = create_data_entries(precursors_df, fragment_columns)
    msms_df = pd.DataFrame(data_entries)
    msms_df.to_csv(os.path.join(SAVE_DIR, f"msms_{file}"), sep='\t')

In [21]:
files_to_process

['0000.tsv',
 '0001.tsv',
 '0002.tsv',
 '0003.tsv',
 '0004.tsv',
 '0005.tsv',
 '0006.tsv',
 '0007.tsv',
 '0008.tsv',
 '0009.tsv',
 '0010.tsv',
 '0011.tsv',
 '0012.tsv',
 '0013.tsv',
 '0014.tsv',
 '0015.tsv',
 '0016.tsv',
 '0017.tsv',
 '0018.tsv',
 '0019.tsv',
 '0020.tsv']